# CrewAI 快速開始教程

## 📚 簡介

CrewAI 是一個角色扮演型多 Agent 協作框架，核心概念：

- **Agent（代理）**: 具有特定角色和能力的 AI 實體
- **Task（任務）**: Agent 需要完成的具體工作
- **Crew（團隊）**: 多個 Agent 組成的協作團隊
- **Process（流程）**: 任務執行的順序和方式

本教程將帶你構建第一個 AI 團隊！

## 1. 安裝 CrewAI

In [ ]:
# 安裝 CrewAI 和工具包
!pip install crewai crewai-tools -q

## 2. 環境配置

In [ ]:
import os
from dotenv import load_dotenv

# 加載環境變數
load_dotenv()

# CrewAI 會自動使用 OPENAI_API_KEY 環境變數
print("環境配置完成！")

## 3. 創建第一個 Agent

讓我們創建一個研究員 Agent：

In [ ]:
from crewai import Agent

# 創建研究員 Agent
researcher = Agent(
    role='AI 研究員',
    goal='深入研究人工智慧領域的最新發展',
    backstory="""你是一位經驗豐富的 AI 研究員，擅長分析技術趨勢，
    閱讀論文和研究報告，並能夠提煉出關鍵洞察。""",
    verbose=True,  # 顯示詳細過程
    allow_delegation=False  # 不允許委派任務
)

print("研究員 Agent 已創建！")

## 4. 創建第二個 Agent

創建一個內容作家 Agent：

In [ ]:
# 創建作家 Agent
writer = Agent(
    role='技術作家',
    goal='撰寫引人入勝且易懂的技術文章',
    backstory="""你是一位專業的技術作家，能夠將複雜的技術概念
    轉化為通俗易懂的內容，讓普通讀者也能理解。""",
    verbose=True,
    allow_delegation=False
)

print("作家 Agent 已創建！")

## 5. 定義任務

為每個 Agent 分配具體任務：

In [ ]:
from crewai import Task

# 研究任務
research_task = Task(
    description="""研究 RAG（檢索增強生成）技術的最新發展。
    請包括：
    1. RAG 的核心概念
    2. 主要應用場景
    3. 最新的技術進展
    4. 面臨的挑戰
    """,
    agent=researcher,
    expected_output="一份詳細的研究報告"
)

# 寫作任務
write_task = Task(
    description="""基於研究結果，撰寫一篇關於 RAG 技術的科普文章。
    要求：
    1. 語言通俗易懂
    2. 結構清晰
    3. 包含實際應用案例
    4. 字數約 500-800 字
    """,
    agent=writer,
    expected_output="一篇完整的科普文章"
)

print("任務已定義！")

## 6. 組建團隊並執行

將 Agent 組成團隊，執行任務：

In [ ]:
from crewai import Crew, Process

# 組建團隊
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,  # 順序執行
    verbose=2  # 最詳細的輸出
)

print("團隊已組建！開始執行任務...\n")

# 執行任務
result = crew.kickoff()

print("\n" + "="*80)
print("執行完成！\n")
print("最終結果：")
print(result)

## 7. 使用工具增強 Agent

為 Agent 添加工具以擴展能力：

In [ ]:
from crewai_tools import SerperDevTool

# 創建搜索工具（需要 SERPER_API_KEY）
# search_tool = SerperDevTool()

# 創建帶工具的研究員
researcher_with_tools = Agent(
    role='高級研究員',
    goal='使用搜索工具研究最新的 AI 技術',
    backstory='你是一位能夠使用網絡搜索工具的高級研究員',
    # tools=[search_tool],  # 取消註釋以使用
    verbose=True,
    allow_delegation=False
)

print("提示：要使用搜索工具，請設置 SERPER_API_KEY 環境變數")

## 8. 簡單實例：技術問答

創建一個簡單的問答系統：

In [ ]:
# 創建問答專家
qa_expert = Agent(
    role='技術問答專家',
    goal='準確回答技術問題',
    backstory='你是一位經驗豐富的技術專家，能夠清晰解答各種技術問題',
    verbose=True
)

# 定義問答任務
qa_task = Task(
    description="請解釋什麼是 LangChain，以及它的主要用途是什麼？",
    agent=qa_expert,
    expected_output="清晰簡潔的解釋"
)

# 執行
qa_crew = Crew(
    agents=[qa_expert],
    tasks=[qa_task],
    verbose=1
)

result = qa_crew.kickoff()
print("\n回答：", result)

## 9. 三人協作範例

創建一個三人團隊：

In [ ]:
# 產品經理
product_manager = Agent(
    role='產品經理',
    goal='規劃產品功能和需求',
    backstory='你是一位經驗豐富的產品經理',
    verbose=False
)

# 開發者
developer = Agent(
    role='軟體開發者',
    goal='實現產品功能',
    backstory='你是一位優秀的全棧開發者',
    verbose=False
)

# QA 測試
qa_tester = Agent(
    role='QA 測試工程師',
    goal='確保產品質量',
    backstory='你是一位細心的測試工程師',
    verbose=False
)

# 定義任務
plan_task = Task(
    description="為一個簡單的待辦事項應用規劃核心功能",
    agent=product_manager,
    expected_output="功能列表"
)

dev_task = Task(
    description="根據功能列表，設計技術實現方案",
    agent=developer,
    expected_output="技術方案"
)

test_task = Task(
    description="制定測試計劃",
    agent=qa_tester,
    expected_output="測試計劃"
)

# 組建開發團隊
dev_crew = Crew(
    agents=[product_manager, developer, qa_tester],
    tasks=[plan_task, dev_task, test_task],
    process=Process.sequential,
    verbose=1
)

result = dev_crew.kickoff()
print("\n開發計劃：", result)

## 📝 總結

本教程中，我們學習了：

1. ✅ CrewAI 的安裝和配置
2. ✅ 創建 Agent（代理）
3. ✅ 定義 Task（任務）
4. ✅ 組建 Crew（團隊）
5. ✅ 執行多 Agent 協作
6. ✅ 使用工具增強能力
7. ✅ 實際應用案例

## 🎯 核心概念

- **Role（角色）**: Agent 的專業領域
- **Goal（目標）**: Agent 要達成的目標
- **Backstory（背景故事）**: Agent 的經驗和特點
- **Process（流程）**: Sequential（順序）或 Hierarchical（階層）

## 🎯 下一步

- 深入學習角色和任務設計：`1.角色和任務.ipynb`
- 探索實際應用案例：`2.實際案例.ipynb`

## 🔗 資源

- [CrewAI 官方文檔](https://docs.crewai.com/)
- [GitHub](https://github.com/joaomdmoura/crewAI)